# BM4Implicit: 48-particle and 24-particle Poincare comparison

This notebook compares two saved BM4Implicit executions at their aligned once-per-cycle returns. The first panel contains 48 particles using 20 steps per cycle; the second contains the 24 odd-numbered particles using 40 steps per cycle. It performs no integration.

Animation, particle selection, square zoom, panning, and point size are synchronized. Drag to zoom; Shift-drag or right-drag to pan; **Full view** restores the periodic cell.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import numpy as np
from IPython.display import FileLink, IFrame, display
from visualization.poincare_comparison import export_poincare_panel_comparison

SOURCES = [
    dict(title="BM4Implicit — 48 particles, 20 steps/cycle", study="Poincare_BM4_48_radiales_5000_ciclos_20_steps_16_procesos_spot", run_id="aws_48p_5000c_20s_16proc_spot_20260920", count=48, steps=20),
    dict(title="BM4Implicit — 24 odd particles, 40 steps/cycle", study="Poincare_BM4Implicit_24_impares_5000_ciclos_40_steps_spot", run_id="aws_bm4implicit_24odd_5000c_40s_20260920", count=24, steps=40),
]
CYCLES = 5000
CYCLES_PER_FRAME = 25
OUTPUT_HTML = Path("bm4implicit_48_vs_24.html")
panels, reference_times = [], None

for source_config in SOURCES:
    folder = Path("..") / source_config["study"] / "resultados" / source_config["run_id"]
    manifest = json.loads((folder / "COMPLETE.json").read_text())
    assert manifest["run_id"] == source_config["run_id"]
    for name in ("metadata.json", "positions_after_each_cycle.csv"):
        path = folder / name
        content = path.read_bytes()
        if content.startswith(b"version https://git-lfs.github.com/spec/v1"):
            raise RuntimeError(f"{path} is a Git LFS pointer. Run `git lfs pull` from the project root.")
        assert hashlib.sha256(content).hexdigest() == manifest["sha256"][name]
    metadata = json.loads((folder / "metadata.json").read_text())
    particle_ids = np.asarray(metadata.get("particle_ids", range(1, metadata["particle_count"] + 1)), dtype=int)
    assert metadata["method"] == "BM4Implicit"
    assert metadata["cycles"] == CYCLES and metadata["steps_per_cycle"] == source_config["steps"]
    assert len(particle_ids) == source_config["count"]
    colors = [metadata["colours"][str(particle_id)] for particle_id in particle_ids]
    coordinates = np.empty((CYCLES, len(particle_ids), 2), dtype=np.float64)
    cycle_times = np.empty((CYCLES, 2), dtype=np.float64)
    with (folder / "positions_after_each_cycle.csv").open(newline="", encoding="utf-8") as handle:
        row_count = 0
        for row_count, row in enumerate(csv.DictReader(handle), start=1):
            assert row_count <= CYCLES * len(particle_ids)
            cycle_index, particle_index = divmod(row_count - 1, len(particle_ids))
            assert int(row["cycle"]) == cycle_index + 1
            assert int(row["particle"]) == particle_ids[particle_index]
            assert row["color"] == colors[particle_index]
            coordinates[cycle_index, particle_index] = (row["x_over_L"], row["y_over_L"])
            row_times = (row["time_normalized"], row["time_s"])
            if particle_index == 0:
                cycle_times[cycle_index] = row_times
            else:
                np.testing.assert_array_equal(np.asarray(row_times, dtype=float), cycle_times[cycle_index])
        assert row_count == CYCLES * len(particle_ids)
    if reference_times is None:
        reference_times = cycle_times
    else:
        np.testing.assert_allclose(cycle_times, reference_times, rtol=1e-13, atol=0)
    panels.append(dict(title=source_config["title"], coordinates=coordinates, particle_ids=particle_ids, colors=colors))
    print(f"{source_config['title']}: {row_count:,} returns loaded")

In [ ]:
export_poincare_panel_comparison(OUTPUT_HTML, panels, cycles_per_frame=CYCLES_PER_FRAME)
print(f"Web visualization available at: {OUTPUT_HTML.resolve()}")
display(FileLink(str(OUTPUT_HTML)))
display(IFrame(str(OUTPUT_HTML), width="100%", height=1050))